In [1]:
# AutoGluon Benchmark - Gutenberg Gait Database

# Imports
import os
import time
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn.metrics
from autogluon.tabular import TabularPredictor

warnings.filterwarnings('ignore')

In [2]:
# Konfiguration
PREPARED_DATA_PATH   = 'saved_models/prepared/prepared_data.csv'
SPLIT_META_PATH      = 'saved_models/prepared/split_meta.json'

# diese Einstellungen werden automatisiert von medium_quality gesetzt:
# PER_RUN_TIME          = 120
# MEMORY_LIMIT_MB       = 8192
# CV_FOLDS              = 5

CLASSIFICATION_TIME  = 300
REGRESSION_TIME      = 300
RANDOM_STATE         = 42
EXCLUDE_WINDOW       = 10

REGRESSION_DIRECTIONS = ['F_V_PRO_', 'F_AP_PRO_', 'F_ML_PRO_']
TARGET_PERCENTAGES    = [20, 40, 60, 80]

AG_PRESETS            = 'medium_quality' #good_quality, high_quality, best_quality
AG_MODEL_DIR          = '/tmp/autogluon_models'

SAVED_MODELS_DIR      = 'saved_models/autogluon'

In [3]:
# Classification-Benchmark: Training + Auswertung
def benchmark_autogluon_classification(feature_df, target_col, groups, split_col, task_name, time_limit):
    print(f"\n{'='*60}")
    print(f"[AutoGluon] Classification Task: {task_name}")
    print(f"  time_limit={time_limit}s")
    print(f"{'='*60}")

    # Split anhand der von 00_preprocessing vorgegebenen 
    train_idx = np.where(split_col == 'train')[0]
    test_idx = np.where(split_col == 'test')[0]
    train_df = feature_df.iloc[train_idx].reset_index(drop=True)
    test_df = feature_df.iloc[test_idx].reset_index(drop=True)

    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
    print(f"Class distribution - Train:\n{train_df[target_col].value_counts()}")
    print(f"Class distribution - Test:\n{test_df[target_col].value_counts()}")

    predictor = TabularPredictor(
        label=target_col,
        problem_type='multiclass' if train_df[target_col].nunique() > 2 else 'binary',
        eval_metric='accuracy',
        path=os.path.join(AG_MODEL_DIR, task_name),
        verbosity=2,
    )

    print("Training AutoGluon classifier...")
    start_time = time.time()
    predictor.fit(train_data=train_df, time_limit=time_limit, presets=AG_PRESETS)
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")

    y_test = test_df[target_col].values
    y_pred = predictor.predict(test_df.drop(columns=[target_col]))

    accuracy = sklearn.metrics.accuracy_score(y_test, y_pred)
    balanced_accuracy = sklearn.metrics.balanced_accuracy_score(y_test, y_pred)
    f1_macro = sklearn.metrics.f1_score(y_test, y_pred, average='macro')
    f1_weighted = sklearn.metrics.f1_score(y_test, y_pred, average='weighted')

    results = {
        'framework': 'AutoGluon',
        'task': task_name,
        'task_type': 'classification',
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'training_time_seconds': training_time,
        'n_train': len(train_df),
        'n_test': len(test_df),
        'classes': int(feature_df[target_col].nunique()),
        'split_method': 'precomputed subject-grouped split (00_preprocessing)',
        'config': {'time_limit': time_limit, 'presets': AG_PRESETS},
    }

    print(f"\nResults for {task_name}:")
    print(f"  Accuracy:          {accuracy:.4f}")
    print(f"  Balanced accuracy: {balanced_accuracy:.4f}")
    print(f"  F1 (macro):        {f1_macro:.4f}")
    print(f"  F1 (weighted):     {f1_weighted:.4f}")
    print(f"  Training time:     {training_time:.2f} seconds")

    return results

In [ ]:
# Regression-Benchmark: Training + Auswertung
def benchmark_autogluon_regression(feature_df, target_col, groups, split_col, task_name, time_limit):
    print(f"\n{'='*60}")
    print(f"[AutoGluon] Regression Task: {task_name}")
    print(f"  time_limit={time_limit}s")
    print(f"{'='*60}")

    y_all = feature_df[target_col].values

    # Split anhand der von 00_preprocessing vorgegebenen 
    train_idx = np.where(split_col == 'train')[0]
    test_idx = np.where(split_col == 'test')[0]
    train_df = feature_df.iloc[train_idx].reset_index(drop=True)
    test_df = feature_df.iloc[test_idx].reset_index(drop=True)

    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
    print(f"Target range - Train: [{train_df[target_col].min():.3f}, {train_df[target_col].max():.3f}]")

    predictor = TabularPredictor(
        label=target_col,
        problem_type='regression',
        eval_metric='r2',
        path=os.path.join(AG_MODEL_DIR, task_name),
        verbosity=2,
    )

    print("Training AutoGluon regressor...")
    start_time = time.time()
    predictor.fit(train_data=train_df, time_limit=time_limit, presets=AG_PRESETS)
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")

    y_test = test_df[target_col].values
    y_pred = predictor.predict(test_df.drop(columns=[target_col])).values

    # Performance metriken berechnen
    r2 = sklearn.metrics.r2_score(y_test, y_pred)
    mae = sklearn.metrics.mean_absolute_error(y_test, y_pred)
    mse = sklearn.metrics.mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_test - y_pred) / np.maximum(np.abs(y_test), 1e-10))) * 100
    y_range = np.ptp(y_test)
    nrmse = rmse / y_range if y_range > 0 else np.nan

    results = {
        'framework': 'AutoGluon',
        'task': task_name,
        'task_type': 'regression',
        'r2': r2,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mape': mape,
        'nrmse': nrmse,
        'training_time_seconds': training_time,
        'n_train': len(train_df),
        'n_test': len(test_df),
        'target_mean': float(np.mean(y_all)),
        'target_std': float(np.std(y_all)),
        'split_method': 'precomputed subject-grouped split (00_preprocessing)',
        'config': {'time_limit': time_limit, 'presets': AG_PRESETS},
    }

    print(f"\nResults for {task_name}:")
    print(f"  R2:    {r2:.4f}")
    print(f"  MAE:   {mae:.4f}")
    print(f"  RMSE:  {rmse:.4f}")
    print(f"  MAPE:  {mape:.2f}% ")
    print(f"  NRMSE: {nrmse:.4f}")
    print(f"  Training time: {training_time:.2f} seconds")

    # Daten + Predictor-Pfad fuer spaetere, separate SHAP-Analyse speichern
    task_dir = os.path.join(SAVED_MODELS_DIR, task_name)
    os.makedirs(task_dir, exist_ok=True)
    train_df.to_csv(os.path.join(task_dir, 'train_data.csv'), index=False)
    test_df.to_csv(os.path.join(task_dir, 'test_data.csv'), index=False)
    feature_names = [c for c in feature_df.columns if c != target_col]
    with open(os.path.join(task_dir, 'feature_names.json'), 'w') as f:
        json.dump(list(feature_names), f)
    with open(os.path.join(task_dir, 'meta.json'), 'w') as f:
        json.dump({'framework': 'AutoGluon', 'task_name': task_name, 'target_col': target_col,
                    'predictor_path': predictor.path, 'split_source': SPLIT_META_PATH}, f)
    print(f"  Daten + Predictor-Pfad fuer SHAP gespeichert unter: {task_dir} "
          f"(Predictor selbst liegt unter: {predictor.path})")

    return results

In [ ]:
# Zentral vorbereitete Daten laden (prepared_data.csv + split_meta.json)
print("="*70)
print("AUTOGLUON BENCHMARK - GUTENBERG GAIT DATABASE (zentraler Split)")
print("="*70)
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nActive config:")
print(f"  CLASSIFICATION_TIME : {CLASSIFICATION_TIME}s")
print(f"  REGRESSION_TIME     : {REGRESSION_TIME}s")
print(f"  REGRESSION_DIRECTIONS: {REGRESSION_DIRECTIONS}")
print(f"  TARGET_PERCENTAGES  : {TARGET_PERCENTAGES}")
print(f"  EXCLUDE_WINDOW      : {EXCLUDE_WINDOW}")
print(f"  AG_PRESETS          : {AG_PRESETS}")

df = pd.read_csv(PREPARED_DATA_PATH)
with open(SPLIT_META_PATH) as f:
    split_meta = json.load(f)

print(f"\nPrepared data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Split: {split_meta['n_subjects_train']} Train-Subjekte, "
      f"{split_meta['n_subjects_test']} Test-Subjekte "
      f"(erzeugt am {split_meta['created_at']}, random_state={split_meta['random_state']})")

required_cols = ['SUBJECT_ID', 'SPLIT', 'SEX_LABEL', 'AGE_BRACKET_LABEL', 'HEIGHT_BRACKET_LABEL']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Erwartete Spalten fehlen in {PREPARED_DATA_PATH}: {missing}. "
                      f"Bitte zuerst 00_preprocessing ausfuehren.")

In [ ]:
# Feature-Basis bauen: alle drei Kraftrichtungen kombiniert (V+AP+ML)
force_columns = []
for prefix in ['F_V_PRO_', 'F_AP_PRO_', 'F_ML_PRO_']:
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        raise ValueError(f"Keine Spalten mit Praefix '{prefix}' gefunden - bitte pruefen, "
                          f"ob 00_preprocessing mit dem AP/ML-Fix gelaufen ist.")
    force_columns.extend(cols)

print(f"Using {len(force_columns)} force features (V+AP+ML kombiniert)")

X = df[force_columns].values
valid_rows = ~np.isnan(X).any(axis=1)
X = X[valid_rows]
print(f"Feature matrix shape: {X.shape}")

df_valid = df[valid_rows].reset_index(drop=True)
groups_all = df_valid['SUBJECT_ID'].values
split_all = df_valid['SPLIT'].values
print(f"Anzahl eindeutiger Subjekte im gueltigen Datensatz: {df_valid['SUBJECT_ID'].nunique()}")

In [ ]:
# Klassifikations-Labels auslesen (bereits zentral kodiert in 00_preprocessing)
labels = {
    'sex': df_valid['SEX_LABEL'].values,
    'age_bracket_encoded': df_valid['AGE_BRACKET_LABEL'].values,
    'height_bracket_encoded': df_valid['HEIGHT_BRACKET_LABEL'].values,
}
print(f"Sex distribution:\n{df_valid['SEX'].value_counts().to_string()}")
print(f"Age bracket distribution:\n{df_valid['AGE_BRACKET'].value_counts().to_string()}")
print(f"Height bracket distribution:\n{df_valid['HEIGHT_BRACKET'].value_counts().to_string()}")

In [8]:
# Ergebnis-Grundstruktur anlegen
all_results = {
    'benchmark_info': {
        'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'n_samples': len(X),
        'n_features': X.shape[1],
        'n_subjects': int(df_valid['SUBJECT_ID'].nunique()),
        'framework': 'AutoGluon',
        'config': {
            'classification_time': CLASSIFICATION_TIME,
            'regression_time': REGRESSION_TIME,
            'regression_directions': REGRESSION_DIRECTIONS,
            'target_percentages': TARGET_PERCENTAGES,
            'exclude_window': EXCLUDE_WINDOW,
            'presets': AG_PRESETS,
            'split_strategy': 'precomputed, shared across frameworks (00_preprocessing)',
            'split_meta_source': SPLIT_META_PATH,
        }
    },
    'results': []
}

In [ ]:
# Klassifikation: Sex, Age-Bracket, Height-Bracket (alle aktiv)
print(f"\n{'#'*60}")
print(f"# CLASSIFICATION BENCHMARKS (AutoGluon)")
print(f"{'#'*60}")

cls_df = pd.DataFrame(X, columns=force_columns)
cls_df['sex'] = labels['sex']
results = benchmark_autogluon_classification(
    cls_df, 'sex', groups_all, split_all, 'sex_classification', time_limit=CLASSIFICATION_TIME,
)
all_results['results'].append(results)

cls_df = pd.DataFrame(X, columns=force_columns)
cls_df['age_bracket_encoded'] = labels['age_bracket_encoded']
results = benchmark_autogluon_classification(
    cls_df, 'age_bracket_encoded', groups_all, split_all, 'age_bracket_classification', time_limit=CLASSIFICATION_TIME,
)
all_results['results'].append(results)

cls_df = pd.DataFrame(X, columns=force_columns)
cls_df['height_bracket_encoded'] = labels['height_bracket_encoded']
results = benchmark_autogluon_classification(
    cls_df, 'height_bracket_encoded', groups_all, split_all, 'height_bracket_classification', time_limit=CLASSIFICATION_TIME,
)
all_results['results'].append(results)

In [ ]:
# Regression: alle Kraftkurven-Ziele (aktiv, 4 Punkte im Gangzyklus x 3 Richtungen)
print(f"\n{'#'*60}")
print(f"# REGRESSION BENCHMARKS - 4 PUNKTE IM GANGZYKLUS x 3 KRAFTRICHTUNGEN")
print(f"# Richtungen: {REGRESSION_DIRECTIONS}")
print(f"# Punkte (% des Zyklus): {TARGET_PERCENTAGES}")
print(f"# (EXCLUDE_WINDOW={EXCLUDE_WINDOW}: benachbarte Spalten derselben Richtung ausgeschlossen)")
print(f"{'#'*60}")

force_results = []
for prefix in REGRESSION_DIRECTIONS:
    cols = sorted((c for c in df_valid.columns if c.startswith(prefix)),
                   key=lambda c: int(c[len(prefix):]))
    n = len(cols)
    for p in TARGET_PERCENTAGES:
        idx = min(max(int(round((p / 100.0) * (n - 1))), 0), n - 1)
        target_col = cols[idx]
        feature_cols = [c for c in cols if c not in
                         set(cols[max(0, idx - EXCLUDE_WINDOW): idx + EXCLUDE_WINDOW + 1])]
        if len(feature_cols) == 0:
            continue

        cols_needed = feature_cols + [target_col, 'SUBJECT_ID', 'SPLIT']
        sub_df = df_valid[cols_needed].dropna()
        if len(sub_df) < 100:
            print(f"  Skipping {target_col}: insufficient samples ({len(sub_df)})")
            continue

        groups_reg = sub_df['SUBJECT_ID'].values
        split_reg = sub_df['SPLIT'].values
        model_input_df = sub_df.drop(columns=['SUBJECT_ID', 'SPLIT'])

        task_name = f"predict_{prefix.rstrip('_')}_{p}pct_{target_col}"
        try:
            results = benchmark_autogluon_regression(
                model_input_df, target_col, groups_reg, split_reg, task_name, time_limit=REGRESSION_TIME,
            )
            results['n_feature_columns'] = len(feature_cols)
            results['exclude_window'] = EXCLUDE_WINDOW
            results['direction'] = prefix
            results['percent_of_cycle'] = p
            force_results.append(results)
        except Exception as e:
            print(f"  Skipping {target_col}: {e}")

all_results['results'].extend(force_results)

In [ ]:
# Regression: demografische Variablen - AGE und HEIGHT (aktiv)
print(f"\n{'#'*60}")
print(f"# REGRESSION BENCHMARKS - DEMOGRAPHIC VARIABLES")
print(f"{'#'*60}")

reg_df = pd.DataFrame(X, columns=force_columns)
reg_df['AGE'] = df_valid['AGE'].values
reg_df['SPLIT'] = split_all
valid_age = ~reg_df['AGE'].isna()
reg_df = reg_df[valid_age].reset_index(drop=True)
groups_age = groups_all[valid_age.values]
split_age = reg_df['SPLIT'].values
reg_df = reg_df.drop(columns=['SPLIT'])
try:
    results = benchmark_autogluon_regression(
        reg_df, 'AGE', groups_age, split_age, 'age_regression', time_limit=REGRESSION_TIME,
    )
    all_results['results'].append(results)
except Exception as e:
    print(f"  Skipping age_regression: {e}")

reg_df = pd.DataFrame(X, columns=force_columns)
reg_df['HEIGHT'] = df_valid['HEIGHT'].values
reg_df['SPLIT'] = split_all
valid_height = ~reg_df['HEIGHT'].isna()
reg_df = reg_df[valid_height].reset_index(drop=True)
groups_height = groups_all[valid_height.values]
split_height = reg_df['SPLIT'].values
reg_df = reg_df.drop(columns=['SPLIT'])
try:
    results = benchmark_autogluon_regression(
        reg_df, 'HEIGHT', groups_height, split_height, 'height_regression', time_limit=REGRESSION_TIME,
    )
    all_results['results'].append(results)
except Exception as e:
    print(f"  Skipping height_regression: {e}")

In [12]:
# Ergebnisse als JSON speichern
output_file = f'autogluon_results_v4_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"Ergebnise gespeichert als {output_file}")

Ergebnise gespeichert als autogluon_results_v4_20260915_205223.json


In [ ]:
# Zusammenfassung ausgeben
print(f"\n{'='*70}")
print("BENCHMARK SUMMARY (AutoGluon, precomputed subject-grouped split)")
print(f"{'='*70}")

tasks = {}
for res in all_results['results']:
    tasks.setdefault(res['task'], []).append(res)

for task, task_results in tasks.items():
    print(f"\n{task}:")
    for res in task_results:
        if res['task_type'] == 'classification':
            print(f"  Accuracy = {res['accuracy']:.4f}  "
                  f"Balanced_Acc = {res['balanced_accuracy']:.4f}  "
                  f"F1_weighted = {res['f1_weighted']:.4f}  "
                  f"Time = {res['training_time_seconds']:.1f}s")
        else:
            print(f"  R2 = {res['r2']:.4f}  "
                  f"MAE = {res['mae']:.4f}  RMSE = {res['rmse']:.4f}  "
                  f"Time = {res['training_time_seconds']:.1f}s")

print(f"\nTotal benchmark completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")